# 02 Data Preprocessing

## Outputs

This notebook writes two files. They show the preprocessing
decisions for the training steps.

| File | Contents | 
|---|---|
| `splits.json` | Image ids for `train`, `val`, `test`, split 70/15/15 from `TrainData`. <br> The split keeps the same proportion of landslide images in all three sets, and uses a fixed seed so it comes out identical every time. | 
| `normalisation_stats.json` | Per-band `mean` and `std` (14 values each), computed on the new train data after splitting. | 

<!-- | `landslide_proportion.npy` | Landslide pixel proportion per image, aligned to ids 1–3799. Cache only — regenerates in ~1 min. | -->


#### Split labelled training data to add labelled test and valid data

In [1]:
import kagglehub
from pathlib import Path

data_dir = Path(kagglehub.dataset_download("tekbahadurkshetri/landslide4sense"))
mask_dir = data_dir / "TrainData" / "mask"

In [2]:
import numpy as np, h5py, json
from pathlib import Path
from sklearn.model_selection import train_test_split

ids = np.arange(1, 3800)
landslide_proportion = np.array([h5py.File(mask_dir / f"mask_{i}.h5")["mask"][:].mean() for i in ids])

tr, tmp = train_test_split(ids, test_size=0.30, stratify=(landslide_proportion > 0), random_state=42)
va, te  = train_test_split(tmp, test_size=0.50, stratify=(landslide_proportion[tmp-1] > 0), random_state=42)

In [3]:
json.dump({"train": tr.tolist(), "val": va.tolist(), "test": te.tolist()},
          open("splits.json", "w"), indent=2)

for name, s in [("train", tr), ("val", va), ("test", te)]:
    print(f"{name:5s} n={len(s):4d}  images with landslide: {(landslide_proportion[s-1] > 0).mean():.3f}  "
          f"landslide pixels: {landslide_proportion[s-1].mean():.4f}")
assert len(set(tr)) + len(set(va)) + len(set(te)) == 3799
assert not (set(tr) & set(va)) and not (set(tr) & set(te)) and not (set(va) & set(te))

train n=2659  images with landslide: 0.587  landslide pixels: 0.0231
val   n= 570  images with landslide: 0.588  landslide pixels: 0.0229
test  n= 570  images with landslide: 0.586  landslide pixels: 0.0239


#### New train split statistics

In [4]:
import json, numpy as np, h5py

img_dir = data_dir / "TrainData" / "img"
train_ids = json.load(open("splits.json"))["train"]

C = 14
sums = np.zeros(C); sqs = np.zeros(C); n = 0
mins = np.full(C, np.inf); maxs = np.full(C, -np.inf)

for i in train_ids:
    with h5py.File(img_dir / f"image_{i}.h5", "r") as f:
        x = f["img"][:].astype(np.float64)
    sums += x.sum(axis=(0, 1))
    sqs  += (x ** 2).sum(axis=(0, 1))
    n    += x.shape[0] * x.shape[1]
    mins = np.minimum(mins, x.min(axis=(0, 1)))
    maxs = np.maximum(maxs, x.max(axis=(0, 1)))

mean = sums / n
std  = np.sqrt(sqs / n - mean ** 2)
z_min = (mins - mean) / std
z_max = (maxs - mean) / std

json.dump({"mean": mean.tolist(), "std": std.tolist(), "n_patches": len(train_ids)},
          open("normalisation_stats.json", "w"), indent=2)

print(f"{'Band':>4} {'mean':>8} {'std':>8} {'z_min':>8} {'z_max':>8}")
for c in range(C):
    print(f"{c+1:4d} {mean[c]:8.3f} {std[c]:8.3f} {z_min[c]:8.1f} {z_max[c]:8.1f}")
print(f"\nNew train statistics, after splitting the data: {len(train_ids)} patches -> normalisation_stats.json")

Band     mean      std    z_min    z_max
   1    0.925    0.140     -6.6     15.6
   2    0.922    0.220     -4.2     85.5
   3    0.955    0.318     -3.0     96.2
   4    0.958    0.571     -1.7     56.4
   5    1.024    0.457     -2.2     19.6
   6    1.049    0.445     -2.4      7.0
   7    1.043    0.465     -2.2      5.7
   8    1.054    0.494     -2.1     14.7
   9    1.172    0.510     -2.3      4.7
  10    1.168    0.677     -1.7     29.9
  11    1.054    0.529     -2.0      7.9
  12    1.039    0.659     -1.6     28.3
  13    1.258    0.673     -1.9      4.1
  14    1.644    1.063     -1.5      3.3

New train statistics, after splitting the data: 2659 patches -> normalisation_stats.json
